# CIS I — Assignment 2: Kinematic Calibration of a Cartesian Robot

This notebook walks through the calibration scenario from the handout
(Figure 1): a cartesian-base + spherical-wrist robot carrying a special
calibration marker body, observed by an optical tracker. Your job is to
**design** the scene (where the tracker and robot go, what the calibration
body's marker layout is), **collect** data under a simulated clock, and
**fit** an improved kinematic model `ImprovedFwdKins` / `ImprovedInvKins`
that beats the nominal one over the robot's work volume: the handout asks
for `||alpha|| <= 1e-3 rad` and `||epsilon|| <= 0.2 mm`, in as little
simulated time as possible — and if you cannot get there, to report the
accuracy you achieved and how you went about it.

Everything runs against `src.simulation`, a self-contained simulator playing
the role of the physical robot + tracker (the tracker and marker-body
classes are assignment 1's, imported from `assignments/assignment1`). It
injects the kinematic errors and measurement noise — your code
never sees ground truth except where explicitly noted for
**validation/reporting** in Steps 8–10.

Unlike assignment 1, your code lives in Python modules under `src/student/`
(every function there is a stub raising `NotImplementedError`), driven from
this notebook and unit-tested from `tests/`. Keep the signatures the
notebook relies on; If you want to design the functions in a different way - feel free to do that. 
But indicate that in your report.
otherwise the design is yours. Remember the assignment's
closing note: this is a design exercise — part of the job is deciding which
items of the tracker's and robot's "spec sheet" matter for your error
budget, and your report must justify those decisions.

In [1]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation, vct3, Rot, Frame
from src.simulation import clock, constants
from src.student import metrics, registration, workflow, kinematics, design

np.set_printoptions(precision=4, suppress=True)
clock.reset()
WALL_START = time.time()        # real run time must stay under 10 minutes when graded

## Step 1: Design the scene

Specify approximate poses (as `Frame`) for the tracking camera and the robot
base, relative to the workspace, and the calibration body's marker layout
(`List[vct3]`, local coordinates). The simulator perturbs the robot's
placement the way the handout says (`||alpha_rob|| <= 0.05`,
`||epsilon_rob|| <= 5 mm`), so `F_rob_approx` is a design value, not truth.

Things to keep in mind:

- The tracker's usable volume (Figure 2) is a frustum along its **local
  +z axis**: 100–220 cm in front of the camera, widening from a 100 cm to a
  200 cm square. `trk.is_visible(points)` checks it for you.
- This handout's tracker has **no jiggle** (`sigma_Ta = sigma_Tp = 0`): the
  pose you give it is exactly where it is, so the tracker frame is a fixed
  reference and everything you measure is expressed in it. What remains is
  0.1 mm of random noise per marker reading and a small systematic field
  (`<= 0.1 mm`, slowly varying).
- The robot's end effector sits 600 mm along the base x-axis at home, and
  the linear joints travel `0..600, -300..300, 0..600` mm — so the work
  volume is a 600 mm cube spanning x ∈ [600, 1200], y ∈ [−300, 300],
  z ∈ [0, 600] in base coordinates, and the wrist (±0.75π, ±0.75π, ±π)
  swings the 75 mm calibration body around inside it. Everything must stay
  visible.
- The calibration body's layout is exact (`a*_k == a_k`) but limited to
  `||a*_k|| <= 75 mm`. Its orientation is estimated from its markers, so
  the number of markers and how far they spread set how noisy each
  orientation reading is.
- You may define and place additional marker bodies
  (`simulation.define_marker_body` / `place_marker_body`) if your design
  wants them; their layouts are only known to ±2 mm (see Step 5).

In [2]:
# Our Step-1 design lives in src/student/design.py (the unit tests use the same values):
#  * tracker 1625 mm above the centre (800, 0, 300) of the 950 mm "marker envelope", looking
#    straight down: the whole envelope lies at depths 1150-2100 mm with >= 87 mm lateral and
#    100 mm depth margin, more than the ~83 mm worst-case shift of the real robot base;
#  * robot base at the workspace origin (its true pose is estimated anyway);
#  * calibration body: 8 markers on the corners of a 75 mm-circumradius cube -- the largest
#    allowed spread, symmetric and non-coplanar: 5.8e-4 rad / 0.035 mm per single reading.
F_tracker_approx = design.tracker_pose()   # Frame, tracker pose relative to the workspace (the camera looks along its local +z)
F_rob_approx = design.robot_pose()         # Frame, robot base pose relative to the workspace
mbee_local_positions = design.mbee_layout()  # 8 points, every ||a*_k|| = 75 mm

print("tracker pose:\n", F_tracker_approx.R.matrix, F_tracker_approx.p)
print("calibration body:", [np.round(a.vec.ravel(), 1).tolist() for a in mbee_local_positions])

tracker pose:
 [[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0. -1.]] vct3(x=800.00, y=0.00, z=1925.00)
Internal Vector:
[[ 800.]
 [   0.]
 [1925.]]
calibration body: [[-43.3, -43.3, -43.3], [-43.3, -43.3, 43.3], [-43.3, 43.3, -43.3], [-43.3, 43.3, 43.3], [43.3, -43.3, -43.3], [43.3, -43.3, 43.3], [43.3, 43.3, -43.3], [43.3, 43.3, 43.3]]


## Step 2: Build the simulation

Fully provided — this wires your Step 1 choices through the simulator's
`CreateRobot` / `CreateTracker` / `CreateRobotCalibrationBody` /
`AddMarkerBodies` equivalents. The robot's hidden kinematic errors and the
tracker's hidden systematic-error field are drawn from `SEED`; your
calibration must work for *any* seed (grading uses different ones), so
don't tune to it.

`mbee_placed` is a `PlacedBody` handle pairing the calibration body with its
hidden actual placement; the body starts attached and moves with the robot.
Note that `rob.state()` already reports non-zero encoder values: the robot
starts at its physical home, `theta* = 0`, and the encoders read
`-delta_theta_sys(home) - delta_theta_rnd` there (see `rob.home_robot()`).

In [3]:
SEED = 1

rob = simulation.create_robot("rob", F_rob_approx, seed=SEED)
trk = simulation.create_tracker("trk", F_tracker_approx, seed=SEED)
mbee_placed = simulation.create_robot_calibration_body(rob, mbee_local_positions)
simulation.add_marker_bodies(trk, [mbee_placed])

print("simulation built:", rob, "|", trk.name, "| tracking", [p.body.name for p in trk.tracked_bodies])
print("robot state:", rob.state())

simulation built: Robot(name='rob', theta_cmd=[-0.0323, -0.0808, 0.0449, -0.0073, -0.0086, 0.0078]) | trk | tracking ['mBee']
robot state: RobotState(theta=array([-0.0323, -0.0808,  0.0449, -0.0073, -0.0086,  0.0078]), theta_dot=array([0., 0., 0., 0., 0., 0.]), t_elapsed=0.0, calibration_body_attached=True)


### Step 2.1: Visibility check

Before spending any simulated time, sweep a joint grid and check that
the calibration body stays inside the tracker's volume (using the *nominal*
placements — they're only tens of mm off, which is nothing against the
frustum size). If a large fraction is invisible, go back to Step 1.

In [4]:
def mbee_markers_world(theta):
    F_ee_world = rob.F_rob * rob.fwd_kins(theta)
    return [F_ee_world * a for a in mbee_local_positions]

n_vis, n_tot = 0, 0
for x in (0, 300, 600):
    for y in (-300, 0, 300):
        for z in (0, 300, 600):
            for w in (-1.8, 0.0, 1.8):
                theta = np.array([x, y, z, w, -w, w])
                n_tot += 1
                n_vis += trk.is_visible(mbee_markers_world(theta))
print(f"calibration body visible at {n_vis}/{n_tot} corner/edge poses of the work volume")

# stronger check: the whole marker envelope (every marker of every reachable pose), shifted by the
# worst-case base placement error (0.05 rad x 1.57 m + 5 mm ~ 83 mm) in every direction
corners = design.envelope_corners(F_rob_approx)
shifts = [vct3(0, 0, 0)] + [vct3(*(85.0 * np.eye(3)[i] * s)) for i in range(3) for s in (1, -1)]
print("envelope visible under all worst-case base shifts:", all(trk.is_visible([c + d for c in corners]) for d in shifts))

calibration body visible at 81/81 corner/edge poses of the work volume
envelope visible under all worst-case base shifts: True


## Step 3: Unit tests

`tests/test_simulation.py` is provided and passes as shipped — read it as
an example. `tests/test_student.py` exercises *your* `src/student` code:
the complete tests there fail with `NotImplementedError` until you
implement what they call, and the ones marked `TODO` are yours to write (you are required to write unit tests of each significant element, described in your
report together with their results). Re-run this cell as you go.

In [5]:
!{sys.executable} -m pytest ../tests -q -p no:cacheprovider

.

.............

.................

....

..

.

... (2 outputs left out)

...

.

.

.                    [100%]
53 passed in 17.29s


## Step 4: Error metric

You have to measure the mismatch between two frames as
`dF = F_a^-1 F_b ≈ Frame(I + sk(alpha), epsilon)`. Implement
`metrics.frame_error` (returning `||alpha||`, `||epsilon||`) and
`metrics.summarize_errors`; every accuracy number below goes through them.

In [6]:
print("sanity:", metrics.frame_error(Frame.eye(), Frame(Rot('z', 0.01), vct3(1, 0, 0))))

sanity: (0.01, 1.0)


## Step 5 (optional): Registration and marker-body refinement

> "It is possible to improve your estimate of a_m,k through a calibration
> process involving multiple observations of the marker body using the
> tracking camera and then averaging them."

The calibration body needs no refinement — `a*_k == a_k` by construction —
so this step only matters if your design places *additional* marker bodies
(their layouts are only known to ±2 mm, and the tracker registers each
body against its *current* nominal layout). You will need your own
`registration.register_points` (Arun's method — do not call the
simulator's solver) and `registration.average_frames` for Step 6 in any
case; `registration.refine_marker_body` is the optional part.

In [7]:
print("register_points sanity:", registration.register_points(
    [vct3(0, 0, 0), vct3(50, 0, 0), vct3(0, 50, 0), vct3(0, 0, 50)],
    [vct3(10, 0, 0), vct3(60, 0, 0), vct3(10, 50, 0), vct3(10, 0, 50)]).p)

register_points sanity: vct3(x=10.00, y=0.00, z=-0.00)
Internal Vector:
[[10.]
 [ 0.]
 [-0.]]


## Step 6: Home the robot, then a baseline in the tracker frame

> "`theta_encoders = HomeRobot()` will place the robot at a home position in
> which `theta*_k = 0` and return the current values read by the encoders.
> This may prove useful as part of your calibration process."

The encoders read `theta = theta* - delta_theta_rnd - delta_theta_sys`, so at
the physical home they expose the systematic offset of every joint
directly. Implement `workflow.measure_home_offsets` (average a few homings).

Then the baseline: the calibration body's coordinate system is exactly
`F_ee`, so every reading satisfies

    F_ee,j  =  F_X · FwdKins(theta_j)

for one constant unknown `F_X` (the robot base expressed in the tracker's
frame). Implement `workflow.collect_calibration_data` (robot move + one
`sample_marker_bodies` reading, storing the body's frame) and
`workflow.estimate_base_frame` (you'll want `registration.average_frames`),
then look at how well the **nominal** kinematics explain even a handful of
poses.

In [8]:
home_offsets = workflow.measure_home_offsets(rob, n_homings=5)   # 5 homings: random part 0.005 mm / 1e-4 rad -> /sqrt(5)
print("delta_theta_sys at home (from the encoders):", home_offsets)

center = np.array([300.0, 0.0, 300.0, 0.0, 0.0, 0.0])
baseline_poses = [center] + [center + np.eye(6)[k] * v for k in range(3) for v in (-250.0, 250.0)] \
                 + [center + np.eye(6)[3] * 0.8, center + np.eye(6)[4] * 0.8, center + np.eye(6)[5] * 1.5]

baseline = workflow.collect_calibration_data(rob, trk, mbee_placed, baseline_poses, repeats=1)
F_X_baseline = workflow.estimate_base_frame(baseline, rob)
residuals = [metrics.frame_error(F_obs, F_X_baseline * rob.fwd_kins(th)) for th, F_obs in zip(baseline.thetas, baseline.frames)]
print(f"{len(baseline)} baseline samples in {baseline.t_collect:.1f}s")
print("nominal kinematics vs observations:", metrics.format_summary(metrics.summarize_errors(residuals)))

delta_theta_sys at home (from the encoders): [ 0.0299  0.0803 -0.0474  0.0074  0.0086 -0.0079]


10 baseline samples in 83.8s
nominal kinematics vs observations: n=10  angle rms 8.19e-03 / max 1.90e-02 rad   pos rms 2.941 / max 5.116 mm   within target 0.0%


## Step 7: Plan and collect the calibration data

> "The goal is to reach a specified accuracy within as short a time as
> possible." — `clock.elapsed` is the score's time component.

The handout suggests three routes and *strongly* urges the first:

1. **Estimate the kinematic parameters.** (a) Work out how closely spaced
   your samples along an axis must be for the accuracy you want — this
   depends on the error bounds `delta_sys^trk`, `omega_sys^trk`,
   `delta_sys^rob`, `omega_sys^rob`; (b) home the robot, then systematically
   move along/about each individual axis while taking data, returning home
   after each axis; (c) determine the angles between the axes and the
   `theta -> theta*` map (use any interpolation scheme you find feasible); (d) write the corrected
   forward and inverse kinematics; (e) report the updated parameters and,
   in debugging, compare them with ground truth; then "bottom line" test a
   sampled portion of the workspace.
2. **Map the space and interpolate** — many more, closely spaced points,
   and dense random-point testing.
3. A mix.

Design `workflow.plan_calibration_poses`: which joint vectors to visit, in
which order. Every parameter of your model must be excited (axis tilts need
travel along each axis; wrist axes, `F_we` and the wrist joints' distortion
need wrist rotations; the linear joints' distortion needs stops along each
joint), every pose must keep the calibration body visible, and the wrist
moves at only 0.1 rad/s. Joints 1–3 are kinematically decoupled from 4–6.

`plan_calibration_poses` takes the accuracy you are aiming for and must
size the plan from it: how many distinct poses, and how many readings of
each (a pose listed twice in a row is re-commanded, costing only `t_settle`
— which also re-rolls the joint repeatability noise — and re-read for
`delta_t_samp`). With 0.1 mm of marker noise on a <= 75 mm body, how many
readings does a 1e-3 rad orientation estimate need? Work it out before you
spend simulated time.

In [9]:
DESIRED_ACCURACY = constants.TARGET_ANG_ERR_RAD  # rad; the orientation target is the limiting one

# Method 1: home, then sweep each joint alone (9 stops per slide, +-1 / +-2 rad per wrist joint).
# The plan sizes itself from DESIRED_ACCURACY: 34 tracker readings per pose give tracker noise
# 1/10 of the orientation budget, taken as 5 re-commands x 7 readings (re-commanding costs only
# t_settle and re-rolls the joint repeatability noise); see workflow.plan_calibration_poses.
poses, plan_info = workflow.plan_calibration_poses(rob, trk, mbee_local_positions,
                                                   desired_accuracy=DESIRED_ACCURACY, return_info=True)
n_distinct = len({tuple(np.round(th, 6)) for th in poses})
print(f"plan: {len(poses)} commands over {n_distinct} distinct poses, {plan_info['commands_per_pose']} commands x "
      f"{plan_info['readings_per_command']} readings = {plan_info['readings_per_pose']} readings per pose")
print(f"      slide step {plan_info['slide_steps_mm'][0]:.0f} mm, wrist stops {plan_info['wrist_stops_rad']} rad, "
      f"predicted {plan_info['predicted_time']['total_s']:.0f} s "
      f"(motion {plan_info['predicted_time']['motion_s']:.0f} s + sampling {plan_info['predicted_time']['sampling_s']:.0f} s)")
print(f"      expected orientation noise per pose: {plan_info['expected_orientation_noise_rad']:.1e} rad")

t0 = clock.elapsed
sweeps = workflow.collect_calibration_data(rob, trk, mbee_placed, poses, repeats=plan_info["readings_per_command"])
print(f"{len(sweeps)} samples ({sum(sweeps.readings)} readings) collected in {sweeps.t_collect:.1f}s "
      f"(predicted {plan_info['predicted_time']['total_s']:.0f}s); clock now {clock.elapsed:.1f}s")

# the Step-6 baseline poses were already paid for: use them in the fit as well
data = workflow.merge_data(baseline, sweeps)
F_X_nom = workflow.estimate_base_frame(data, rob)

plan: 200 commands over 37 distinct poses, 5 commands x 7 readings = 35 readings per pose
      slide step 75 mm, wrist stops [-2.0, -1.0, 1.0, 2.0] rad, predicted 305 s (motion 263 s + sampling 42 s)
      expected orientation noise per pose: 9.8e-05 rad


39 samples (1400 readings) collected in 304.9s (predicted 305s); clock now 389.1s


## Step 8: Fit the improved kinematics

This is the assignment's core deliverable — implement
`kinematics.ImprovedKinematics` yourself and describe it in the report.

The handout's error model: `theta*_k = theta_k + delta_theta_k,rnd +
delta_theta_k,sys` on every joint (smooth, `||delta_theta_sys|| <=
delta_sys`, slope `<= omega_sys`), `d*_k = R(phi_k) d_k` for `k = 2..6`
(`d*_1 = d_1`), `F*_we = F_we dF_ee`, plus the base frame `F_X`. Choose a
parameter vector (the class docstring sketches one), write
`fwd_kins(theta, params)`, a residual against `data`, and a non-linear
least-squares `fit` (`scipy.optimize` is fine; so is a numerical
Jacobian). `inv_kins` is provided: it iterates on *your* `fwd_kins`.

**Questions for the report** (your `jacobian_rank_report` should help):
is every parameter identifiable from your poses? Is every parameter in the
handout's model identifiable *at all* — what happens to a component of
`phi_k` along `d_k`, to a rotation of the base about `d_1`, or to a joint
offset on joints 1–3 versus a shift of the base? (Which is why the home
offsets come from `HomeRobot()` rather than from the fit.) Which of the
handout's error terms are big enough to matter at the 0.2 mm / 1e-3 rad
level, and which are below the noise? How did that shape your
parameterization and your pose plan?

**Ground truth for debugging only**: `rob.ground_truth_parameters()`
returns the hidden kinematic parameters so you can compare your
`parameters_report()` with them, as the handout asks for debugging runs —
never use it in the calibration itself.

In [10]:
# Parameterization: base twist (6) + axis tilts (2 x 5) + tool twist (6) + distortion polynomials
# (degree 2 on the slides, 1 on the wrist: the wrist ripple has slope <= 5e-5 rad/rad, a quadratic
# only fits noise). Joint offsets come from HomeRobot() and are not fitted. The base rotation
# about d_1 is frozen (fix_gauge=True): it is the model's one unobservable direction.
improved = kinematics.ImprovedKinematics(rob, F_X_nom, joint_offsets=home_offsets, n_poly=(2, 2, 2, 1, 1, 1))
fit_info = improved.fit(data)
print("fit:", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in fit_info.items()})

rank = improved.jacobian_rank_report(data)
print("jacobian rank, all parameters:", rank["rank"], "/", rank["n_params"],
      "| weakest direction:", rank["weakest_directions"][0]["direction"])
print("jacobian rank, fitted parameters:", rank["free"]["rank"], "/", rank["free"]["n_params"],
      f"| condition number {rank['free']['condition_number']:.1f}")

t_calibration = clock.elapsed
print(f"total simulated calibration time: {t_calibration:.1f}s (soft budget {constants.TARGET_TIME_SEC:.0f}s)")

estimated, truth = improved.parameters_report(), rob.ground_truth_parameters()   # truth: debugging only
cmp = improved.compare_with_ground_truth(truth, trk.nominal_pose())
print("estimated axis angles (deg):", estimated["axis_angles_deg"], estimated["axis_angle_pairs"])
print("   true axis angles (deg):", cmp["axis_angles_deg_true"], "| error (mrad):", np.radians(cmp["axis_angles_deg_err"]) * 1e3)
print("estimated joint offsets:", estimated["joint_offset"], "| true (at home):", cmp["joint_offset_true_at_home"])
print("estimated joint scales :", estimated["joint_scale"], "| true:", truth["joint_scale"])
print("axis direction errors after removing the base gauge (mrad):", cmp["axis_direction_err_mrad"])
print("tool frame error (rad, mm):", cmp["tool_frame_err_rad_mm"], "| base frame error (rad, mm):", cmp["base_frame_err_rad_mm"])

fit: {'status': 2, 'message': '`ftol` termination condition is satisfied.', 'nfev': 5, 'cost': 0.257756, 'n_samples': 49, 'n_free_params': 30, 'rms_rot_residual_rad': 0.000492, 'rms_pos_residual_mm': 0.028873, 'max_pos_residual_mm': 0.076716, 'fix_gauge': True, 'fixed_params': ['base_alpha_x']}


jacobian rank, all parameters: 30 / 31 | weakest direction: [('tilt_axis2_x', 0.408), ('tool_alpha_x', 0.408), ('tilt_axis3_x', 0.408), ('base_alpha_x', -0.408), ('tilt_axis6_x', 0.408), ('tilt_axis5_x', 0.408)]
jacobian rank, fitted parameters: 30 / 30 | condition number 72.7
total simulated calibration time: 389.1s (soft budget 900s)
estimated axis angles (deg): [90.0717 90.1852 88.3065 91.8101 90.1757 89.3803] ['d1-d2', 'd1-d3', 'd2-d3', 'n4-n5', 'n4-n6', 'n5-n6']
   true axis angles (deg): [90.0685 90.1799 88.308  91.8122 90.1719 89.3844] | error (mrad): [ 0.0558  0.0924 -0.0267 -0.0378  0.0661 -0.0718]
estimated joint offsets: [ 0.0299  0.0803 -0.0474  0.0074  0.0086 -0.0079] | true (at home): [ 0.0346  0.0796 -0.0487  0.0075  0.0086 -0.0079]
estimated joint scales : [-0.0002 -0.0002 -0.0003 -0.      0.0001  0.    ] | true: [-0.0002 -0.0002 -0.0002 -0.      0.     -0.    ]
axis direction errors after removing the base gauge (mrad): [0.0403 0.0687 0.067  0.0797 0.0856 0.0938]
tool 

## Step 9: Evaluate `ImprovedFwdKins` over the work volume

**Ground truth for reporting only**: `rob.actual_ee_pose(theta)` (the
actual end-effector pose in the workspace for an encoder command `theta`)
is a hidden value you have access to here only because this notebook
doesn't gate it. Use it *only* in Steps 9–10 to compute and report your
achieved error. With no tracker jiggle, the tracker's true pose is the one
you specified, so `F_trk = trk.nominal_pose()` maps between the workspace
and the tracker frame.

The metric, per test pose `theta_i`, with `F̂_X` your estimated base frame:

    dF_i = (F_trk^-1 · F*_ee(theta_i))^-1 · (F̂_X · ImprovedFwdKins(theta_i))

Report `||alpha||` and `||epsilon||` statistics over the grid below (which
is fixed — don't change it) for the nominal kinematics and for yours, and
compare against `constants.TARGET_POS_ERR_MM` / `TARGET_ANG_ERR_RAD`.
Evaluation reads the oracle only; it must not move the robot or sample the
tracker.

In [11]:
test_levels = {0: (100.0, 300.0, 500.0), 1: (-200.0, 0.0, 200.0), 2: (100.0, 300.0, 500.0),
               3: (-1.3, 0.0, 1.3), 4: (-1.3, 0.0, 1.3), 5: (-2.0, 0.0, 2.0)}
test_grid = [np.array([x, y, z, a, b, c])
             for x in test_levels[0] for y in test_levels[1] for z in test_levels[2]
             for a in test_levels[3] for b in test_levels[4] for c in test_levels[5]]
test_grid = [th for th in test_grid if trk.is_visible(mbee_markers_world(th))]
print(f"{len(test_grid)} visible test poses")

F_trk = trk.nominal_pose()

fwd_err_nominal, fwd_err_improved = [], []
for theta in test_grid:
    truth_pose = F_trk.inv() * rob.actual_ee_pose(theta)            # ground truth, reporting only
    fwd_err_nominal.append(metrics.frame_error(truth_pose, F_X_nom * rob.fwd_kins(theta)))
    fwd_err_improved.append(metrics.frame_error(truth_pose, improved.predict_in_tracker(theta)))

print("ImprovedFwdKins :", metrics.format_summary(metrics.summarize_errors(fwd_err_improved)))
print("nominal FwdKins :", metrics.format_summary(metrics.summarize_errors(fwd_err_nominal)))

729 visible test poses


ImprovedFwdKins : n=729  angle rms 2.11e-04 / max 3.48e-04 rad   pos rms 0.080 / max 0.118 mm   within target 100.0%
nominal FwdKins : n=729  angle rms 3.29e-02 / max 5.60e-02 rad   pos rms 7.070 / max 16.104 mm   within target 0.0%


## Step 10: Evaluate `ImprovedInvKins`

For a target pose `F` (in the tracker frame), `theta =
ImprovedInvKins(F̂_X^-1 F)`, then actually move the robot there and compare
where it ends up — `mbee_placed.actual_placement` after the move, relative to
`F_trk` — with the target. Use targets you know are reachable (e.g.
ground-truth poses of test-grid joint vectors). The handout asks for the
time required for testing as well as for calibration, so keep track of it.
Note that every real move includes the joint repeatability noise
`sigma_rnd`; think about what floor that puts on this number.

In [12]:
rng = np.random.default_rng(0)
targets = [test_grid[i] for i in rng.choice(len(test_grid), size=min(40, len(test_grid)), replace=False)]
# same 40 targets, visited in a greedy nearest-neighbour order: the wrist moves at 0.1 rad/s, so the
# visiting order is what sets the testing time (the handout asks us to report that time)
targets = workflow.order_poses_by_motion_time(rob, targets)
t_test_start = clock.elapsed

inv_err_nominal, inv_err_improved = [], []
for theta_t in targets:
    F_target = F_trk.inv() * rob.actual_ee_pose(theta_t)   # a reachable target, in the tracker frame
    for kins, F_X, errs in ((rob, F_X_nom, inv_err_nominal), (improved, improved.base_frame(), inv_err_improved)):
        theta = kins.inv_kins(F_X.inv() * F_target, seed=theta_t)
        rob.move_joints(theta)
        actual = F_trk.inv() * mbee_placed.actual_placement           # ground truth, reporting only
        errs.append(metrics.frame_error(F_target, actual))

t_testing = clock.elapsed - t_test_start
print("ImprovedInvKins :", metrics.format_summary(metrics.summarize_errors(inv_err_improved)))
print("nominal InvKins :", metrics.format_summary(metrics.summarize_errors(inv_err_nominal)))
print(f"simulated testing time: {t_testing:.1f}s")

ImprovedInvKins : n=40  angle rms 2.92e-04 / max 4.94e-04 rad   pos rms 0.083 / max 0.115 mm   within target 100.0%
nominal InvKins : n=40  angle rms 3.45e-02 / max 5.59e-02 rad   pos rms 7.432 / max 13.274 mm   within target 0.0%
simulated testing time: 481.5s


## Achieved performance

Summarize: nominal vs improved, forward and inverse, against the handout's
targets (0.2 mm, 1e-3 rad), plus the simulated time spent calibrating and
testing and how the calibration time was spent (homing, number of
moves/samples). If a target is not met, say what you achieved and why —
that is an acceptable outcome per the handout. Put the same table in your
report, together with results for at least one other `SEED`.

In [13]:
def row(name, s):
    ok = s["max_pos"] <= constants.TARGET_POS_ERR_MM and s["max_angle"] <= constants.TARGET_ANG_ERR_RAD
    return (f"| {name:16s} | {s['rms_pos']:.3f} | {s['max_pos']:.3f} | {s['rms_angle']:.2e} | {s['max_angle']:.2e} | "
            f"{100 * s['frac_within_target']:5.1f}% | {'yes' if ok else 'NO'} |")

summary = {
    "fwd_nominal": metrics.summarize_errors(fwd_err_nominal), "fwd_improved": metrics.summarize_errors(fwd_err_improved),
    "inv_nominal": metrics.summarize_errors(inv_err_nominal), "inv_improved": metrics.summarize_errors(inv_err_improved),
}
print(f"SEED = {SEED}; targets: {constants.TARGET_POS_ERR_MM} mm, {constants.TARGET_ANG_ERR_RAD} rad")
print("| model            | pos rms | pos max | ang rms | ang max | in spec | meets target |")
print("|------------------|---------|---------|---------|---------|---------|--------------|")
for name, key in (("nominal FwdKins", "fwd_nominal"), ("ImprovedFwdKins", "fwd_improved"),
                  ("nominal InvKins", "inv_nominal"), ("ImprovedInvKins", "inv_improved")):
    print(row(name, summary[key]))
print(f"\nsimulated time: calibration {t_calibration:.1f} s "
      f"(homing {5 * rob.t_settle:.1f} s + baseline {baseline.t_collect:.1f} s + sweeps {sweeps.t_collect:.1f} s, "
      f"{len(poses)} commands, {sum(sweeps.readings) + sum(baseline.readings)} tracker readings), "
      f"testing {t_testing:.1f} s; real run time so far {time.time() - WALL_START:.0f} s")

SEED = 1; targets: 0.2 mm, 0.001 rad
| model            | pos rms | pos max | ang rms | ang max | in spec | meets target |
|------------------|---------|---------|---------|---------|---------|--------------|
| nominal FwdKins  | 7.070 | 16.104 | 3.29e-02 | 5.60e-02 |   0.0% | NO |
| ImprovedFwdKins  | 0.080 | 0.118 | 2.11e-04 | 3.48e-04 | 100.0% | yes |
| nominal InvKins  | 7.432 | 13.274 | 3.45e-02 | 5.59e-02 |   0.0% | NO |
| ImprovedInvKins  | 0.083 | 0.115 | 2.92e-04 | 4.94e-04 | 100.0% | yes |

simulated time: calibration 389.1 s (homing 0.2 s + baseline 83.8 s + sweeps 304.9 s, 200 commands, 1410 tracker readings), testing 481.5 s; real run time so far 24 s


## Repeat for other seeds

The handout and the README ask for results for more than one robot. `run_pipeline(seed)`
repeats Steps 2-10 on a fresh robot and tracker (the same code as above, nothing tuned to the seed)
and returns the summary table; the report takes its tables from `results/notebook_results.json`.

In [14]:
def run_pipeline(seed, n_inv_targets=40, verbose=False):
    clock.reset()
    rob_s = simulation.create_robot("rob", F_rob_approx, seed=seed)
    trk_s = simulation.create_tracker("trk", F_tracker_approx, seed=seed)
    mbee_s = simulation.create_robot_calibration_body(rob_s, mbee_local_positions)
    simulation.add_marker_bodies(trk_s, [mbee_s])
    offsets = workflow.measure_home_offsets(rob_s, n_homings=5)
    base = workflow.collect_calibration_data(rob_s, trk_s, mbee_s, baseline_poses, repeats=1)
    poses_s, info_s = workflow.plan_calibration_poses(rob_s, trk_s, mbee_local_positions, desired_accuracy=DESIRED_ACCURACY, return_info=True)
    sw = workflow.collect_calibration_data(rob_s, trk_s, mbee_s, poses_s, repeats=info_s["readings_per_command"])
    d = workflow.merge_data(base, sw)
    FX0 = workflow.estimate_base_frame(d, rob_s)
    imp = kinematics.ImprovedKinematics(rob_s, FX0, joint_offsets=offsets, n_poly=(2, 2, 2, 1, 1, 1))
    fit_s = imp.fit(d)
    t_cal = clock.elapsed
    Ft = trk_s.nominal_pose()
    grid = [th for th in test_grid]   # same fixed grid (visibility does not depend on the seed)
    fn, fi = [], []
    for th in grid:
        tp = Ft.inv() * rob_s.actual_ee_pose(th)
        fn.append(metrics.frame_error(tp, FX0 * rob_s.fwd_kins(th)))
        fi.append(metrics.frame_error(tp, imp.predict_in_tracker(th)))
    r = np.random.default_rng(0)
    tg = workflow.order_poses_by_motion_time(rob_s, [grid[i] for i in r.choice(len(grid), size=min(n_inv_targets, len(grid)), replace=False)])
    t0 = clock.elapsed
    inn, ini = [], []
    for th_t in tg:
        Ftar = Ft.inv() * rob_s.actual_ee_pose(th_t)
        for kins, FX, errs in ((rob_s, FX0, inn), (imp, imp.base_frame(), ini)):
            rob_s.move_joints(kins.inv_kins(FX.inv() * Ftar, seed=th_t))
            errs.append(metrics.frame_error(Ftar, Ft.inv() * mbee_s.actual_placement))
    t_tst = clock.elapsed - t0
    cmp_s = imp.compare_with_ground_truth(rob_s.ground_truth_parameters(), Ft)
    out = {"seed": seed, "t_calibration": t_cal, "t_testing": t_tst, "t_baseline": base.t_collect, "t_sweeps": sw.t_collect,
           "n_commands": len(poses_s), "readings_per_pose": info_s["readings_per_pose"],
           "readings_per_command": info_s["readings_per_command"], "commands_per_pose": info_s["commands_per_pose"], "fit": fit_s,
           "fwd_nominal": metrics.summarize_errors(fn), "fwd_improved": metrics.summarize_errors(fi),
           "inv_nominal": metrics.summarize_errors(inn), "inv_improved": metrics.summarize_errors(ini),
           "axis_direction_err_mrad": cmp_s["axis_direction_err_mrad"].tolist(),
           "axis_angles_deg_err": cmp_s["axis_angles_deg_err"].tolist(), "joint_scale_err": cmp_s["joint_scale_err"].tolist(),
           "joint_offset_err": cmp_s["joint_offset_err"].tolist(), "tool_frame_err_rad_mm": list(cmp_s["tool_frame_err_rad_mm"]),
           "base_frame_err_rad_mm": list(cmp_s["base_frame_err_rad_mm"])}
    if verbose:
        print(f"seed {seed}: ImprovedFwdKins {metrics.format_summary(out['fwd_improved'])}")
        print(f"         ImprovedInvKins {metrics.format_summary(out['inv_improved'])}")
        print(f"         t_cal {t_cal:.0f} s, t_test {t_tst:.0f} s")
    return out

# the notebook's own run (Steps 6-10 above) is the first entry; seeds 2 and 3 are fresh runs of the same code
main_entry = {"seed": SEED, "t_calibration": t_calibration, "t_testing": t_testing, "t_baseline": baseline.t_collect,
              "t_sweeps": sweeps.t_collect, "n_commands": len(poses), "readings_per_pose": plan_info["readings_per_pose"],
              "readings_per_command": plan_info["readings_per_command"], "commands_per_pose": plan_info["commands_per_pose"],
              "fit": fit_info, "fwd_nominal": summary["fwd_nominal"], "fwd_improved": summary["fwd_improved"],
              "inv_nominal": summary["inv_nominal"], "inv_improved": summary["inv_improved"],
              "axis_direction_err_mrad": cmp["axis_direction_err_mrad"].tolist(), "axis_angles_deg_err": cmp["axis_angles_deg_err"].tolist(),
              "joint_scale_err": cmp["joint_scale_err"].tolist(), "joint_offset_err": cmp["joint_offset_err"].tolist(),
              "tool_frame_err_rad_mm": list(cmp["tool_frame_err_rad_mm"]), "base_frame_err_rad_mm": list(cmp["base_frame_err_rad_mm"])}
results = {"notebook_seed": SEED, "seeds": [main_entry], "plan_info": {k: v for k, v in plan_info.items() if k != "slide_stops"},
           "fit_info": fit_info, "rank": {"rank": rank["rank"], "n_params": rank["n_params"],
                                          "weakest": rank["weakest_directions"][0], "free_rank": rank["free"]["rank"],
                                          "free_n": rank["free"]["n_params"], "free_cond": rank["free"]["condition_number"],
                                          "singular_values": rank["singular_values"]},
           "params_vs_truth": {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in cmp.items()},
           "test_grid_size": len(test_grid)}
for s in (2, 3):
    results["seeds"].append(run_pipeline(s, verbose=True))
print(f"\nreal run time for the whole notebook: {time.time() - WALL_START:.0f} s")

os.makedirs("../results", exist_ok=True)
with open("../results/notebook_results.json", "w") as f:
    json.dump(results, f, indent=1, default=lambda o: o.tolist() if isinstance(o, np.ndarray) else (float(o) if isinstance(o, np.floating) else str(o)))
print("saved ../results/notebook_results.json")

seed 2: ImprovedFwdKins n=729  angle rms 2.35e-04 / max 4.45e-04 rad   pos rms 0.062 / max 0.108 mm   within target 100.0%
         ImprovedInvKins n=40  angle rms 2.91e-04 / max 6.09e-04 rad   pos rms 0.066 / max 0.098 mm   within target 100.0%
         t_cal 389 s, t_test 475 s


seed 3: ImprovedFwdKins n=729  angle rms 3.47e-04 / max 5.72e-04 rad   pos rms 0.065 / max 0.109 mm   within target 100.0%
         ImprovedInvKins n=40  angle rms 3.72e-04 / max 7.42e-04 rad   pos rms 0.066 / max 0.088 mm   within target 100.0%
         t_cal 389 s, t_test 458 s

real run time for the whole notebook: 35 s
saved ../results/notebook_results.json


## Pose-plan / time trade-off

The same pipeline with deliberately different plans (wrist sweep length, slide spacing, readings
per pose, re-commanding, polynomial degrees), each on 10 robot seeds, is run by
`report/variant_study.py` (kept out of this notebook to keep its run time short; it takes about
6 minutes) and written to `results/variants.json`. The report's trade-off table comes from there.